In [71]:
from langchain_core.messages import HumanMessage, SystemMessage
from typing_extensions import TypedDict
from enum import Enum, auto
from langchain_core.prompts import ChatPromptTemplate
from langchain_openrouter import ChatOpenRouter
from langgraph.graph import START, END, StateGraph
from pydantic import BaseModel
from collections import Counter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

In [72]:
class FeedbackColor(Enum):
    GREEN = auto()
    YELLOW = auto()
    GREY = auto()

class Guess(TypedDict):
    guess: str
    feedback: list[FeedbackColor]

class GameState(TypedDict):
    word: str
    guesses: list[Guess]
    remaining_guesses: int

class GuessOut(BaseModel):
    guess:str

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()  # reads ./.env into os.environ

# llm = ChatOpenRouter(
#     model="nvidia/nemotron-3-nano-30b-a3b:free",
#     api_key=os.environ["OPENROUTER_API_KEY"],
#     reasoning= {"reasoning": {"enabled": True}}
# )

# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     api_key=os.environ["GOOGLE_API_KEY"],
#     thinking_budget=0,
#     # reasoning={"enabled": True}
# )

llm = ChatOpenAI(
    model="mlx-community/Qwen3-4b-4bit",
    base_url="http://localhost:8001/v1",
    api_key="EMPTY",
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

In [74]:
def format_guesses(guesses: list[Guess]) -> str:
    if not guesses:
        return "No guesses yet — this is your first attempt."

    verdict = {
        FeedbackColor.GREEN:  "GREEN  (correct letter, correct position — LOCKED here)",
        FeedbackColor.YELLOW: "YELLOW (correct letter, wrong position — must appear elsewhere)",
        FeedbackColor.GREY:   "GREY   (letter NOT in the answer)",
    }

    blocks = []
    for i, g in enumerate(guesses, 1):
        word = g["guess"].upper()
        lines = [f"Guess {i}: {word}"]
        for pos, (ch, fb) in enumerate(zip(word, g["feedback"]), 1):
            lines.append(f"  pos {pos}: {ch}  ->  {verdict[fb]}")
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks)

def score_guess(guess: str, answer: str) -> list[FeedbackColor]:
    guess = guess.lower()
    answer = answer.lower()

    if len(guess) != len(answer):
        return [FeedbackColor.GREY] * len(answer)

    feedback = [FeedbackColor.GREY] * len(answer)
    remaining = Counter(answer)

    for i, ch in enumerate(guess):
        if ch == answer[i]:
            feedback[i] = FeedbackColor.GREEN
            remaining[ch] -= 1

    for i, ch in enumerate(guess):
        if feedback[i] is FeedbackColor.GREEN:
            continue
        if remaining[ch] > 0:
            feedback[i] = FeedbackColor.YELLOW
            remaining[ch] -= 1

    return feedback

def should_guess(state: GameState) -> str:
    last = state["guesses"][-1] if state["guesses"] else None
    if last and all(c == FeedbackColor.GREEN for c in last["feedback"]):
        return "won"
    if state["remaining_guesses"] <= 0:
        return "lost"
    return "continue"

def feedback_node(state: GameState):
    last = state["guesses"][-1]
    scored: Guess = {
        "guess": last["guess"],
        "feedback": score_guess(last["guess"], state["word"])
    }
    state["guesses"][-1] = scored
    print(scored)

    return {
        "guesses": state["guesses"],
        "remaining_guesses": state["remaining_guesses"] - 1
    }

MAX_RETRIES = 5

# ---------- Agent 1: Baseline ----------
baseline_guess_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
        "You are playing Wordle with 5-letter words. "
        "Each prior guess is shown letter-by-letter with a verdict per position: "
        "GREEN (correct letter, correct position), "
        "YELLOW (correct letter, wrong position), "
        "GREY (letter NOT in the answer). "
        "Use prior feedback to narrow down the answer. Never repeat a guess. "
        "Never reuse a letter that was marked GREY. "
        "Reply with JSON only: {{\"guess\": \"<5 letters, lowercase>\"}}"),
        ("human",
        "History:\n{history}\n\n"
        "Forbidden (already guessed, do NOT repeat): {forbidden}\n\n"
        "Make your next guess — it MUST differ from every forbidden word."),
    ])
    | llm.with_structured_output(GuessOut, method="json_schema")
)

def baseline_guess_node(state: GameState):
    prior = {g["guess"].lower() for g in state["guesses"]}
    forbidden_str = ", ".join(sorted(prior)) if prior else "(none)"
    history = format_guesses(state["guesses"])

    guess_str = ""
    for _ in range(MAX_RETRIES):
        out = baseline_guess_chain.invoke({"history": history, "forbidden": forbidden_str})
        candidate = out.guess.lower()
        if candidate not in prior:
            guess_str = candidate
            break
        print(f"[baseline duplicate {candidate!r}, retrying]")
    else:
        guess_str = candidate

    new_guess: Guess = {"guess": guess_str, "feedback": []}
    return {"guesses": state["guesses"] + [new_guess]}


In [75]:
import re, json

# ---------- Agent 2: Chain-of-Thought (prompted, not schema-enforced) ----------
cot_prompt = ChatPromptTemplate.from_messages([
    ("system",
    "You are playing Wordle with 5-letter words. "
    "Each prior guess is shown letter-by-letter with a verdict per position: "
    "GREEN (correct letter, correct position), "
    "YELLOW (correct letter, wrong position), "
    "GREY (letter NOT in the answer). "
    "Reason out loud BEFORE guessing. Walk through:\n"
    "  - Confirmed letters: GREEN letters and the exact positions they are locked in.\n"
    "  - Misplaced letters: YELLOW letters and the positions they CANNOT occupy "
    "(the position where they were guessed).\n"
    "  - Eliminated letters: GREY letters — these are NOT in the answer.\n"
    "  - Candidate words consistent with all of the above, and why your pick is best.\n"
    "Never repeat a previous guess. Never reuse an eliminated (GREY) letter.\n"
    "After your reasoning, end your message with EXACTLY this line and nothing after it:\n"
    "GUESS: <5-letter-word-lowercase>"),
    ("human",
    "History:\n{history}\n\n"
    "Forbidden (already guessed, do NOT repeat): {forbidden}\n\n"
    "Reason through the constraints, then give your guess."),
])

cot_chain = cot_prompt | llm  # plain LLM call — model produces free-form CoT + final line

def _parse_cot_guess(text: str) -> str:
    # Preferred: a final "GUESS: xxxxx" line
    m = re.search(r"GUESS\s*:\s*([a-zA-Z]{5})\b", text)
    if m:
        return m.group(1).lower()
    # Fallback: a JSON object with a "guess" field
    m = re.search(r'\{[^{}]*"guess"\s*:\s*"([a-zA-Z]{5})"[^{}]*\}', text)
    if m:
        return m.group(1).lower()
    # Last resort: last 5-letter token in the response
    tokens = re.findall(r"\b([a-zA-Z]{5})\b", text)
    return tokens[-1].lower() if tokens else ""

def cot_guess_node(state: GameState):
    prior = {g["guess"].lower() for g in state["guesses"]}
    forbidden_str = ", ".join(sorted(prior)) if prior else "(none)"
    history = format_guesses(state["guesses"])

    guess_str = ""
    last_text = ""
    for _ in range(MAX_RETRIES):
        msg = cot_chain.invoke({"history": history, "forbidden": forbidden_str})
        last_text = msg.content or ""
        candidate = _parse_cot_guess(last_text)
        if candidate and candidate not in prior:
            guess_str = candidate
            break
        print(f"[cot duplicate-or-empty {candidate!r}, retrying]")
    else:
        guess_str = candidate

    print(f"[CoT reasoning]:\n{last_text.strip()}\n[CoT -> {guess_str!r}]")

    new_guess: Guess = {"guess": guess_str, "feedback": []}
    return {"guesses": state["guesses"] + [new_guess]}


In [76]:
# ---------- Agent 3: Self-Reflection ----------
propose_prompt = ChatPromptTemplate.from_messages([
    ("system",
    "You are playing Wordle with 5-letter words. "
    "Each prior guess is shown with per-position verdicts: GREEN (locked here), "
    "YELLOW (in word but not at this position), GREY (not in the answer). "
    "Propose ONE candidate 5-letter word consistent with all prior feedback. "
    "Never repeat a previous guess. Never use a GREY letter. "
    "Reply with JSON only: {{\"guess\": \"<5 letters, lowercase>\"}}"),
    ("human",
    "History:\n{history}\n\n"
    "Forbidden (already guessed): {forbidden}\n\n"
    "Previous attempt: {prior_attempt}\n"
    "Critique of previous attempt: {critique}\n\n"
    "Propose your next candidate."),
])
propose_chain = propose_prompt | llm.with_structured_output(GuessOut, method="json_schema")

# Critic with prompted chain-of-thought (no schema enforcement, like the CoT agent).
critique_prompt = ChatPromptTemplate.from_messages([
    ("system",
    "You audit a candidate Wordle guess against known constraints from prior feedback. "
    "Each prior guess shows per-position verdicts: GREEN (locked at that position), "
    "YELLOW (in the word, but NOT at that position), GREY (not in the answer at all).\n\n"
    "Reason step by step BEFORE your verdict. Walk through, in order:\n"
    "  1. List every GREEN: each (letter, position) that is LOCKED. If none, say \"no greens\".\n"
    "  2. List every YELLOW: each (letter, position) where that letter must appear elsewhere "
    "in the word but NOT at that position. If none, say \"no yellows\".\n"
    "  3. List every GREY letter — letters that are NOT in the answer. If none, say \"no greys\".\n"
    "  4. Check the candidate against each constraint one by one. State pass or fail for each.\n"
    "  5. Conclude.\n\n"
    "Hard rules:\n"
    "  - Only cite constraints actually present in the history. Do NOT invent greens, "
    "yellows, or greys that the history does not show.\n"
    "  - If the history is empty, every candidate is automatically VALID.\n"
    "  - A candidate is INVALID only if it fails at least one constraint above.\n\n"
    "After your reasoning, end your message with EXACTLY one of these forms and nothing after:\n"
    "VERDICT: VALID\n"
    "  --or--\n"
    "VERDICT: INVALID\n"
    "VIOLATIONS: <one short line listing the real failures>"),
    ("human",
    "History:\n{history}\n\n"
    "Candidate: {candidate}\n\n"
    "Reason through each constraint, then give your verdict."),
])
critique_chain = critique_prompt | llm

def _parse_critique(text: str) -> tuple[bool, str]:
    m = re.search(r"VERDICT\s*:\s*(VALID|INVALID)", text, re.IGNORECASE)
    if not m:
        return True, ""
    if m.group(1).upper() == "VALID":
        return True, ""
    vm = re.search(r"VIOLATIONS\s*:\s*(.+)", text, re.IGNORECASE)
    return False, (vm.group(1).strip() if vm else "unspecified violations")

MAX_REFLECTIONS = 3

def reflection_guess_node(state: GameState):
    prior = {g["guess"].lower() for g in state["guesses"]}
    forbidden_str = ", ".join(sorted(prior)) if prior else "(none)"
    history = format_guesses(state["guesses"])

    prior_attempt = "(none)"
    critique_text = "(none yet)"
    candidate = ""

    for attempt in range(MAX_REFLECTIONS):
        out = propose_chain.invoke({
            "history": history,
            "forbidden": forbidden_str,
            "prior_attempt": prior_attempt,
            "critique": critique_text,
        })
        candidate = out.guess.lower()

        if candidate in prior:
            print(f"[reflection attempt {attempt+1}] duplicate {candidate!r} -- forcing revision")
            prior_attempt = candidate
            critique_text = f"'{candidate}' is in the forbidden list — pick a different word."
            continue

        msg = critique_chain.invoke({"history": history, "candidate": candidate})
        crit_text = msg.content or ""
        is_valid, violations = _parse_critique(crit_text)

        print(f"[reflection attempt {attempt+1}] candidate={candidate!r}")
        print(f"  critic reasoning:\n{crit_text.strip()}")
        print(f"  -> {'VALID' if is_valid else f'INVALID ({violations})'}")

        if is_valid:
            break

        prior_attempt = candidate
        critique_text = violations

    new_guess: Guess = {"guess": candidate, "feedback": []}
    return {"guesses": state["guesses"] + [new_guess]}


In [77]:
# ---------- Agent 4: Self-Consistency ----------
sc_llm = ChatOpenAI(
    model="mlx-community/Qwen3-4b-4bit",
    base_url="http://localhost:8001/v1",
    api_key="EMPTY",
    temperature=1.3,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

self_consistency_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
        "You are playing Wordle with 5-letter words. "
        "Use the per-position GREEN/YELLOW/GREY history to pick a 5-letter word. "
        "Never repeat a previous guess. Never use a GREY letter. "
        "Reply JSON only: {{\"guess\": \"<5 letters, lowercase>\"}}"),
        ("human", "History:\n{history}\n\nForbidden: {forbidden}"),
    ])
    | sc_llm.with_structured_output(GuessOut, method="json_schema")
)

def count_constraints_satisfied(candidate: str, guesses: list[Guess]) -> int:
    cand = candidate.lower()
    score = 0
    for g in guesses:
        for i, (ch, fb) in enumerate(zip(g["guess"].lower(), g["feedback"])):
            if fb == FeedbackColor.GREEN  and i < len(cand) and cand[i] == ch:    score += 1
            elif fb == FeedbackColor.YELLOW and ch in cand and (i >= len(cand) or cand[i] != ch): score += 1
            elif fb == FeedbackColor.GREY  and ch not in cand:                    score += 1
    return score

def self_consistency_guess_node(state: GameState):
    prior = {g["guess"].lower() for g in state["guesses"]}
    forbidden = ", ".join(sorted(prior)) or "(none)"
    history = format_guesses(state["guesses"])

    inputs = [{"history": history, "forbidden": forbidden}] * 3
    candidates = [o.guess.lower() for o in self_consistency_chain.batch(inputs)]
    print(f"[self-consistency] candidates: {candidates}")

    counts = Counter(candidates)
    top, top_count = counts.most_common(1)[0]
    if top_count >= 2:
        chosen = top
    else:
        chosen = max(candidates, key=lambda c: count_constraints_satisfied(c, state["guesses"]))
    print(f"[self-consistency] picked: {chosen!r}")

    new_guess: Guess = {"guess": chosen, "feedback": []}
    return {"guesses": state["guesses"] + [new_guess]}


In [78]:
def build_app(guess_node_fn):
    g = StateGraph(GameState)
    g.add_node("guess", guess_node_fn)
    g.add_node("feedback", feedback_node)
    g.add_edge(START, "guess")
    g.add_edge("guess", "feedback")
    g.add_conditional_edges("feedback", should_guess, {
        "continue": "guess",
        "won": END,
        "lost": END,
    })
    return g.compile()

baseline_app         = build_app(baseline_guess_node)
cot_app              = build_app(cot_guess_node)
reflection_app       = build_app(reflection_guess_node)
self_consistency_app = build_app(self_consistency_guess_node)

In [79]:
WORDS = ["crane", "light", "proud", "storm", "blaze"]

def run_experiment(app, words):
    rows = []
    for word in words:
        print(f"\n--- word: {word!r} ---")
        r = app.invoke({"word": word, "guesses": [], "remaining_guesses": 6})
        last = r["guesses"][-1] if r["guesses"] else None
        solved = bool(last) and all(c == FeedbackColor.GREEN for c in last["feedback"]) \
                            and last["guess"].lower() == word.lower()
        rows.append({
            "word": word,
            "solved": solved,
            "guesses_used": len(r["guesses"]),
            "sequence": [g["guess"].upper() for g in r["guesses"]],
        })
    return rows

def print_agent_results(agent_name, rows):
    print(f"\n===== {agent_name} =====")
    print(f"{'Word':<10}{'Solved':<10}{'#Guesses':<12}Sequence")
    print("-" * 70)
    for r in rows:
        print(f"{r['word']:<10}{str(r['solved']):<10}{r['guesses_used']:<12}"
              f"{' -> '.join(r['sequence'])}")
    wins = sum(1 for r in rows if r["solved"])
    won_rows = [r for r in rows if r["solved"]]
    avg_g = (sum(r["guesses_used"] for r in won_rows) / len(won_rows)) if won_rows else float("nan")
    print(f"\nWins: {wins}/{len(rows)} ({100*wins/len(rows):.0f}%)  "
          f"avg guesses on wins: {avg_g:.2f}")

In [80]:
# ----- BASELINE on all words -----
baseline_rows = run_experiment(baseline_app, WORDS)
print_agent_results("Baseline", baseline_rows)


--- word: 'crane' ---
{'guess': 'crane', 'feedback': [<FeedbackColor.GREEN: 1>, <FeedbackColor.GREEN: 1>, <FeedbackColor.GREEN: 1>, <FeedbackColor.GREEN: 1>, <FeedbackColor.GREEN: 1>]}

--- word: 'light' ---
{'guess': 'crane', 'feedback': [<FeedbackColor.GREY: 3>, <FeedbackColor.GREY: 3>, <FeedbackColor.GREY: 3>, <FeedbackColor.GREY: 3>, <FeedbackColor.GREY: 3>]}
{'guess': 'bloss', 'feedback': [<FeedbackColor.GREY: 3>, <FeedbackColor.YELLOW: 2>, <FeedbackColor.GREY: 3>, <FeedbackColor.GREY: 3>, <FeedbackColor.GREY: 3>]}
{'guess': 'flies', 'feedback': [<FeedbackColor.GREY: 3>, <FeedbackColor.YELLOW: 2>, <FeedbackColor.YELLOW: 2>, <FeedbackColor.GREY: 3>, <FeedbackColor.GREY: 3>]}
{'guess': 'strip', 'feedback': [<FeedbackColor.GREY: 3>, <FeedbackColor.YELLOW: 2>, <FeedbackColor.GREY: 3>, <FeedbackColor.YELLOW: 2>, <FeedbackColor.GREY: 3>]}
{'guess': 'stirr', 'feedback': [<FeedbackColor.GREY: 3>, <FeedbackColor.YELLOW: 2>, <FeedbackColor.YELLOW: 2>, <FeedbackColor.GREY: 3>, <FeedbackColo

In [81]:
# ----- CHAIN-OF-THOUGHT on all words -----
cot_rows = run_experiment(cot_app, WORDS)
print_agent_results("Chain-of-Thought", cot_rows)


--- word: 'crane' ---
[CoT reasoning]:
Since this is the first attempt and no letters have been guessed yet, there are no confirmed letters, misplaced letters, or eliminated letters. Therefore, there are no constraints to eliminate words or letters. All 5-letter words are still candidates.

To start, I can choose any 5-letter word. I'll select "crane" as it is a common and valid 5-letter word.

GUESS: crane
[CoT -> 'crane']
{'guess': 'crane', 'feedback': [<FeedbackColor.GREEN: 1>, <FeedbackColor.GREEN: 1>, <FeedbackColor.GREEN: 1>, <FeedbackColor.GREEN: 1>, <FeedbackColor.GREEN: 1>]}

--- word: 'light' ---
[CoT reasoning]:
Since this is the first attempt, there are no confirmed letters, misplaced letters, or eliminated letters. Therefore, there are no constraints to apply. All 5-letter words are still candidates, and no letters have been eliminated.

The best approach is to choose a word that is common and likely to have a mix of letters that can help identify the answer in subsequent

KeyboardInterrupt: 

In [ ]:
# ----- SELF-REFLECTION on all words -----
reflection_rows = run_experiment(reflection_app, WORDS)
print_agent_results("Self-Reflection", reflection_rows)

In [ ]:
# ----- SELF-CONSISTENCY on all words -----
self_consistency_rows = run_experiment(self_consistency_app, WORDS)
print_agent_results("Self-Consistency", self_consistency_rows)

In [ ]:
# ----- Cross-agent win % summary -----
all_runs = [
    ("Baseline",         baseline_rows),
    ("Chain-of-Thought", cot_rows),
    ("Self-Reflection",  reflection_rows),
    ("Self-Consistency", self_consistency_rows),
]

print(f"{'Agent':<20}{'Wins':<8}{'Total':<8}{'Win %':<10}{'Avg # guesses (wins only)'}")
print("-" * 70)
for name, rows in all_runs:
    wins = sum(1 for r in rows if r["solved"])
    total = len(rows)
    won = [r for r in rows if r["solved"]]
    avg_g = (sum(r["guesses_used"] for r in won) / len(won)) if won else float("nan")
    print(f"{name:<20}{wins:<8}{total:<8}{100*wins/total:<10.0f}{avg_g:.2f}")